# Retrieve evidence and abstain when it is absent

Prerequisites: Tokenization, vectors, cosine similarity, and source provenance.

Retrieval selects candidate evidence; generation turns context into an answer. Those are different stages with different failure modes. This companion implements lexical TF-IDF retrieval over four fictional documents and returns quoted passages with document IDs. It performs no language-model generation and does not verify a passage's truth merely because its score is high.

Each token's inverse-document-frequency weight grows when it appears in fewer documents. Cosine similarity compares normalized vectors. The query recovery data age should rank the recovery passage first. A query with no corpus tokens has a zero vector; the implementation returns an explicit abstention instead of dividing by zero or fabricating evidence.

One document contains a quoted malicious instruction. It remains inert text in the corpus. The retrieval function has no tool executor, credentials or network capability. In a larger RAG system, retrieved text must still be treated as untrusted data; good retrieval scores are neither authorization nor a defense against injection. Evaluate retrieval recall separately from the factual support of any generated answer.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Index | Four local text fixtures | Known corpus and IDs |
| Retriever | TF-IDF cosine | Lexical ranking |
| Response | Quoted passages or abstain | No generated claim of truth |

## Predict

The default query returns d1 first. An out-of-vocabulary query such as xyzzy produces no evidence and an abstention. Searching for disclose secrets may retrieve d4, but must not perform the quoted instruction.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import ai_models as m

In [ ]:
result=m.retrieval()
assert result['retrieved'][0]['id']=='d1'
assert not m.retrieval('xyzzy')['retrieved']
assert m.retrieval('disclose secrets')['retrieved'][0]['id']=='d4'
print(result)

## Perturb and explain

Create five labeled queries including one missing answer and one adversarial passage. Record recall@k and unsupported-answer rate as separate metrics.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

A high retrieval score establishes:

1. That a document is true
2. That a tool request is authorized
3. Similarity under the selected representation

Answer and explanations: 1: Similarity is not fact checking. 2: Retrieved content carries no execution authority. 3: The score measures the defined matching function.

## Primary evidence

- [Lewis et al. — Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [OWASP — LLM Prompt Injection Prevention Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/LLM_Prompt_Injection_Prevention_Cheat_Sheet.html)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.